<h2>Impact (v0.5) - 'Impact (v0.5).ipynb'</h2>

The aim of the 'Impact' notebooks are to address the following questions laid out in Q1.1.4 of 'Working Paper 1' focusing on 'Task 1.1 - GFI Trajectories and Relevance to African Shipping'.

<u><h4>How do the fleet-level transport cost outputs generated in '3. LEAP III Inventories (v0.1).ipynb' map to commodity flow volumes and GDP-level welfare effects?</h4></u>

We will first focus on the mapping to commodity flows, for which we will need seaborne trade portfolios as a first step. Once we have these, we can then proceed with the mapping step between fleet-level transport cost outputs and commodity flows. To achieve this, we will first integrate Sea Distances derived in the 'CERDI Sea Distances' table. We can then derive the product of the sea distance with the cargo volume, and multiply this by the transport cost intensities identified in the matching '3. LEAP III Inventories (v0.1).ipynb' notebook.

<u><h4> How sensitive are these estimates to data availability for African trade routes where vessel-tracking data is incomplete?</h4></u>

The seaborne trade portfolios are here derived from UNCTAD's Trade-and-Transport database. This primarily owes to the fact that the database is able to provide bilateral seaborne trade portfolios 'off-the-shelf' to users, however there are limitations with the data within the dataset that principally owe to the quality of input Comtrade data utilised for the year in question. Care should therefore taken to explore and understand the quality associated with the Trade-and-Transport-derived seaborne trade portfolio derived.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios.

Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.5' is amended to work with Comtrade import and export trade portfolios downloaded from the platform.

Limitations of the method include the following:

<ol type=1>
    <li>There are inherent limitations when using the Trade-and-Transport dataset, further details of which can be found its underlying documentation.</li>
    <li>2018 data is used, which is increasingly out of date and may not reflect the latest changes impacting each of the country's of interest.</li>
</ol>

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# Define functions for plotting a count of values in a given data field.
int_arr_voys_field_count = lambda field: int_arr_voys[field].value_counts().to_frame().reset_index()
int_dep_voys_field_count = lambda field: int_dep_voys[field].value_counts().to_frame().reset_index()

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.5"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"

# 1. Set up the notebook

## 1.1 Voyages with FEM Results

In [2]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv("/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_0.1.csv", dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 1.2 Let's set the country we're interested in and set up a results folder.

In [3]:
# Select one of our countries to work on...
iso_code = "388"
iso_country = "Jamaica"
year = 2024
gdp = 22.010e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

'Output directory already exists.'

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_v0.6.csv".format(iso_country), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.netWgt.sum() / 1e6, 1)))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Jamaica': 

Number of trade records present in dataset: 11585
Number of partner countries present in dataset: 116
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$6836.5m
Total weight within the dataset: 5865.1 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,388,Jamaica,752,Sweden,000,World,Import,Sea,"Ethyl alcohol, undenatured; of an alcoholic st...",2208,l,73029.6,85544.16,491034.75,NaN,491034.75,"Ethyl alcohol, unden","2208 | Ethyl alcohol, unden","Sweden | 2208 | Ethyl alcohol, unden",22,"22 | Beverages, spirits a","Beverages, spirits a","Sweden | 22 | Beverages, spirits a","[Container, Chemical tanker]",7110.0,4417.947810,377929.634330
1,1,388,Jamaica,528,Netherlands,000,World,Import,Sea,"Waters, including mineral and aerated waters, ...",2202,l,233592.8,253045.76,618397.24,NaN,618397.24,"Waters, including mi","2202 | Waters, including mi","Netherlands | 2202 | Waters, including mi",22,"22 | Beverages, spirits a","Beverages, spirits a","Netherlands | 22 | Beverages, spirits a",[Container],6261.0,3890.403831,984450.194122


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 2198
Total size of the fleet: 757 vessels
8 vessel types contained in the dataset: ['Bulk carrier' 'General cargo' 'Chemical tanker' 'Refrigerated bulk'
 'Vehicle' 'Container' 'Liquefied gas tanker' 'Oil tanker']
Total transport work capacity: 147495.7 million deadweight-tonne nautical miles
Total cargo transported: 46008452.6 tonnes
Total transport work: 85425.7 million tonne nautical miles
Average build year of the fleet: 2007
Total Energy Demands: 15374.3 EJ
Total CO2 produced: 1183883.2 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 2.3 Evaluate NZF Costs

In [14]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_arr_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_arr_voys_vess.shape[0] == 0:
                int_arr_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_arr_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
            
            if int_arr_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
                continue
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["lh_levy_usd_"+year] = lh_levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.netWgt.sum() / 1e6, 1)))


print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_im.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.lh_levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_na = trade_im[(trade_im[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im = trade_im[(~trade_im[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['8806 | Unmanned aircraft' '9620 | Monopods, bipods, tr'
 '8549 | Electrical and elect']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

[]


Number of trade records present in dataset: 11585
Number of partner countries present in dataset: 116

Total value of trade: US$6836.5m
Total weight within the dataset: 5865.1 million kilograms


Total FEM costs in 2041...

BAU: US$139.8m (2.0% ad-val)
Levy: US$294.2m (4.3% ad-val)
Levy (low-then-high): US$287.0m (4.2% ad-val)
'as-is': US$277.8m (4.1% ad-val)
Brazil: US$286.6m (4.2% ad-val)
LISCR: US$191.3m (2.8% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,388,Jamaica,752,Sweden,000,World,Import,Sea,"Ethyl alcohol, undenatured; of an alcoholic st...",2208,l,73029.6,85544.16,491034.75,NaN,491034.75,"Ethyl alcohol, unden","2208 | Ethyl alcohol, unden","Sweden | 2208 | Ethyl alcohol, unden",22,"22 | Beverages, spirits a","Beverages, spirits a","Sweden | 22 | Beverages, spirits a","[Container, Chemical tanker]",7110.0,4417.947810,377929.634330,2676.206278,2669.729214,2670.031779,2670.335146,2670.335146,2673.704475,2801.500921,5005.778998,3181.952254,3049.925478,2846.463821,2856.079943,2742.171735,4864.319811,3587.934964,3230.353452,2964.816171,2897.375323,2690.150990,5394.803123,4974.158750,3929.232540,3510.424677,3255.001071,2639.027171,5726.854470,5396.206195,4630.362411,4325.304296,3447.176281,2588.860994,5737.319675,5664.354158,5575.605263,5584.705933,3723.044997,2537.701310,5690.628430,5576.665349,5426.955105,5864.982024,4476.684334,2489.519550,4997.566996,5159.131905,5297.799788,5685.122565,4277.994058,2440.432266,4561.988837,4643.343267,4638.500619,5542.935120,4572.421079
1,1,388,Jamaica,528,Netherlands,000,World,Import,Sea,"Waters, including mineral and aerated waters, ...",2202,l,233592.8,253045.76,618397.24,NaN,618397.24,"Waters, including mi","2202 | Waters, including mi","Netherlands | 2202 | Waters, including mi",22,"22 | Beverages, spirits a","Beverages, spirits a","Netherlands | 22 | Beverages, spirits a",[Container],6261.0,3890.403831,984450.194122,6751.261327,6733.005140,6733.854703,6734.706441,6734.706441,6744.361399,7087.555046,13017.117034,8102.421430,7765.733410,7235.790611,7261.497299,6928.284855,12658.493111,9200.707138,8205.434482,7527.318895,7341.599371,6788.640914,14010.415753,12794.232798,10157.482193,8954.974721,8261.212818,6651.372771,14769.165853,13985.696382,11922.943891,11109.381821,8786.618255,6516.672392,15002.154227,14780.165420,14570.886391,14477.736877,9534.101029,6379.310127,14838.844221,14512.980158,14077.150560,15355.701524,11603.653211,6249.950325,12964.099904,13446.454134,13846.216002,14772.267304,10988.499424,6118.179746,11780.279915,12032.109266,11968.831456,14414.178725,11853.869688


## 2.4 Inpsect Output

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,149.6,146.8,144.4,142.1,139.8,137.4,135.1,132.7
1,Levy,252.0,244.4,268.8,293.4,294.2,281.4,253.6,234.7
2,Levy (LH),165.5,184.0,260.4,277.9,287.0,281.7,263.1,237.0
3,'as-is',159.5,167.2,200.3,234.6,277.8,270.8,263.9,241.1
4,Brazil,150.6,156.2,182.3,220.6,286.6,293.0,287.2,279.9
5,LISCR,151.3,153.0,170.0,178.3,191.3,226.9,223.0,229.2



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.2,2.1,2.1,2.1,2.0,2.0,2.0,1.9
1,Levy,3.7,3.6,3.9,4.3,4.3,4.1,3.7,3.4
2,Levy (LH),2.4,2.7,3.8,4.1,4.2,4.1,3.8,3.5
3,'as-is',2.3,2.4,2.9,3.4,4.1,4.0,3.9,3.5
4,Brazil,2.2,2.3,2.7,3.2,4.2,4.3,4.2,4.1
5,LISCR,2.2,2.2,2.5,2.6,2.8,3.3,3.3,3.4



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.680,0.667,0.656,0.645,0.635,0.624,0.614,0.603
1,Levy,1.145,1.110,1.221,1.333,1.337,1.279,1.152,1.066
2,Levy (LH),0.752,0.836,1.183,1.262,1.304,1.280,1.195,1.077
3,'as-is',0.725,0.760,0.910,1.066,1.262,1.230,1.199,1.095
4,Brazil,0.684,0.710,0.828,1.002,1.302,1.331,1.305,1.272
5,LISCR,0.687,0.695,0.773,0.810,0.869,1.031,1.013,1.041


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,102.3,97.6,124.4,151.3,154.5,144.0,118.5,102.0
1,Levy (LH),15.8,37.2,116.0,135.8,147.3,144.4,127.9,104.3
2,'as-is',9.9,20.4,55.9,92.5,138.1,133.5,128.8,108.4
3,Brazil,1.0,9.5,37.9,78.5,146.8,155.7,152.1,147.2
4,LISCR,1.6,6.2,25.7,36.2,51.5,89.5,87.9,96.5



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.5,1.4,1.8,2.2,2.3,2.1,1.7,1.5
1,Levy (LH),0.2,0.5,1.7,2.0,2.2,2.1,1.9,1.5
2,'as-is',0.1,0.3,0.8,1.4,2.0,2.0,1.9,1.6
3,Brazil,0.0,0.1,0.6,1.1,2.1,2.3,2.2,2.2
4,LISCR,0.0,0.1,0.4,0.5,0.8,1.3,1.3,1.4



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.465,0.443,0.565,0.688,0.702,0.654,0.538,0.464
1,Levy (LH),0.072,0.169,0.527,0.617,0.669,0.656,0.581,0.474
2,'as-is',0.045,0.093,0.254,0.420,0.627,0.606,0.585,0.492
3,Brazil,0.004,0.043,0.172,0.357,0.667,0.707,0.691,0.669
4,LISCR,0.007,0.028,0.117,0.164,0.234,0.407,0.399,0.438


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_v0.6.csv".format(iso_country), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.netWgt.sum() / 1e6, 1)))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Jamaica': 

Number of trade records present in dataset: 2331
Number of partner countries present in dataset: 81
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1821.5m
Total weight within the dataset: 4869.0 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,388,Jamaica,214,Dominican Rep.,000,World,Export,Sea,"Hydrometers and similar floating instruments, ...",9025,NaN,0.0,5.50,NaN,276,276,Hydrometers and simi,9025 | Hydrometers and simi,Dominican Rep. | 9025 | Hydrometers and simi,90,"90 | Optical, photographi","Optical, photographi","Dominican Rep. | 90 | Optical, photographi",[Container],732.0,454.843572,2.50164
1,1,388,Jamaica,659,Saint Kitts and Nevis,000,World,Export,Sea,"Centrifuges, including centrifugal dryers; fil...",8421,NaN,0.0,0.77,NaN,1237,1237,"Centrifuges, includi","8421 | Centrifuges, includi","Saint Kitts and Nevis | 8421 | Centrifuges, in...",84,84 | Machinery and mechan,Machinery and mechan,Saint Kitts and Nevis | 84 | Machinery and mechan,[Container],1392.0,864.948432,0.66601


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 2206
Total size of the fleet: 758 vessels
8 vessel types contained in the dataset: ['Bulk carrier' 'General cargo' 'Chemical tanker' 'Refrigerated bulk'
 'Vehicle' 'Container' 'Liquefied gas tanker' 'Oil tanker']
Total transport work capacity: 147878.0 million deadweight-tonne nautical miles
Total cargo transported: 46138312.1 tonnes
Total transport work: 88561.5 million tonne nautical miles
Average build year of the fleet: 2007
Total Energy Demands: 14091.0 EJ
Total CO2 produced: 1075556.0 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_dep_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_dep_voys_vess.shape[0] == 0:
                int_dep_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_dep_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

            if int_dep_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["lh_levy_usd_"+year] = lh_levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.netWgt.sum() / 1e6, 1)))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.lh_levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_na = trade_ex[(trade_ex[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(
    trade_ex_na.primaryValue.sum(), 
    round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)
))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['8806 | Unmanned aircraft' '9620 | Monopods, bipods, tr'
 '8549 | Electrical and elect']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

[]


Number of trade records present in dataset: 2331
Number of partner countries present in dataset: 81

Total value of trade: US$1821.5m
Total weight within the dataset: 4869.0 million kilograms


Total FEM costs in 2041...

BAU: US$69.9m (3.8% ad-val)
Levy: US$138.3m (7.6% ad-val)
Levy (low-then-high): US$137.1m (7.5% ad-val)
'as-is': US$131.8m (7.2% ad-val)
Brazil: US$136.8m (7.5% ad-val)
LISCR: US$96.3m (5.3% ad-val)
Rows with NA: $0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,388,Jamaica,214,Dominican Rep.,000,World,Export,Sea,"Hydrometers and similar floating instruments, ...",9025,NaN,0.0,5.50,NaN,276,276,Hydrometers and simi,9025 | Hydrometers and simi,Dominican Rep. | 9025 | Hydrometers and simi,90,"90 | Optical, photographi","Optical, photographi","Dominican Rep. | 90 | Optical, photographi",[Container],732.0,454.843572,2.50164,0.017168,0.017121,0.017124,0.017126,0.017126,0.017150,0.018023,0.033104,0.020605,0.019749,0.018400,0.018465,0.017618,0.032188,0.023393,0.020865,0.019141,0.018669,0.017263,0.035625,0.032545,0.025831,0.022771,0.021007,0.016914,0.037556,0.035557,0.030318,0.028250,0.022343,0.016571,0.038149,0.037579,0.037045,0.036814,0.024244,0.016222,0.037732,0.036910,0.035798,0.039043,0.029500,0.015893,0.032969,0.034198,0.035203,0.037552,0.027934,0.015558,0.029957,0.030597,0.030438,0.036652,0.030142
1,1,388,Jamaica,659,Saint Kitts and Nevis,000,World,Export,Sea,"Centrifuges, including centrifugal dryers; fil...",8421,NaN,0.0,0.77,NaN,1237,1237,"Centrifuges, includi","8421 | Centrifuges, includi","Saint Kitts and Nevis | 8421 | Centrifuges, in...",84,84 | Machinery and mechan,Machinery and mechan,Saint Kitts and Nevis | 84 | Machinery and mechan,[Container],1392.0,864.948432,0.66601,0.004571,0.004558,0.004559,0.004559,0.004559,0.004566,0.004798,0.008813,0.005486,0.005258,0.004899,0.004916,0.004690,0.008569,0.006228,0.005555,0.005096,0.004970,0.004596,0.009484,0.008664,0.006877,0.006062,0.005593,0.004503,0.009999,0.009466,0.008072,0.007521,0.005948,0.004412,0.010156,0.010005,0.009862,0.009801,0.006454,0.004319,0.010045,0.009827,0.009530,0.010394,0.007854,0.004231,0.008777,0.009105,0.009372,0.009997,0.007437,0.004142,0.007976,0.008146,0.008104,0.009758,0.008025


## 3.4 Inspect Output

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,73.9,72.8,71.8,70.8,69.9,68.9,68.0,67.0
1,Levy,115.3,111.7,122.6,134.7,138.3,129.6,120.3,113.2
2,Levy (LH),80.7,87.3,124.6,134.3,137.1,131.7,125.1,113.9
3,'as-is',77.7,80.4,95.6,108.9,131.8,126.5,122.7,115.4
4,Brazil,73.7,76.8,88.6,103.4,136.8,133.7,133.5,130.5
5,LISCR,73.9,75.7,83.9,87.7,96.3,110.8,108.1,110.7



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,4.1,4.0,3.9,3.9,3.8,3.8,3.7,3.7
1,Levy,6.3,6.1,6.7,7.4,7.6,7.1,6.6,6.2
2,Levy (LH),4.4,4.8,6.8,7.4,7.5,7.2,6.9,6.3
3,'as-is',4.3,4.4,5.2,6.0,7.2,6.9,6.7,6.3
4,Brazil,4.0,4.2,4.9,5.7,7.5,7.3,7.3,7.2
5,LISCR,4.1,4.2,4.6,4.8,5.3,6.1,5.9,6.1



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.336,0.331,0.326,0.322,0.317,0.313,0.309,0.305
1,Levy,0.524,0.508,0.557,0.612,0.628,0.589,0.547,0.514
2,Levy (LH),0.367,0.396,0.566,0.610,0.623,0.599,0.568,0.518
3,'as-is',0.353,0.366,0.434,0.495,0.599,0.575,0.558,0.524
4,Brazil,0.335,0.349,0.403,0.470,0.621,0.608,0.607,0.593
5,LISCR,0.336,0.344,0.381,0.399,0.437,0.503,0.491,0.503


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,41.4,38.9,50.8,63.9,68.4,60.7,52.4,46.2
1,Levy (LH),6.8,14.5,52.8,63.5,67.2,62.8,57.1,46.9
2,'as-is',3.8,7.7,23.8,38.0,62.0,57.6,54.8,48.4
3,Brazil,-0.2,4.0,16.8,32.6,66.9,64.8,65.5,63.5
4,LISCR,-0.0,3.0,12.1,16.9,26.4,41.9,40.1,43.7



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,2.3,2.1,2.8,3.5,3.8,3.3,2.9,2.5
1,Levy (LH),0.4,0.8,2.9,3.5,3.7,3.5,3.1,2.6
2,'as-is',0.2,0.4,1.3,2.1,3.4,3.2,3.0,2.7
3,Brazil,-0.0,0.2,0.9,1.8,3.7,3.6,3.6,3.5
4,LISCR,-0.0,0.2,0.7,0.9,1.4,2.3,2.2,2.4



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.188,0.177,0.231,0.290,0.311,0.276,0.238,0.210
1,Levy (LH),0.031,0.066,0.240,0.288,0.305,0.286,0.260,0.213
2,'as-is',0.017,0.035,0.108,0.173,0.282,0.262,0.249,0.220
3,Brazil,-0.001,0.018,0.076,0.148,0.304,0.295,0.298,0.288
4,LISCR,-0.000,0.013,0.055,0.077,0.120,0.190,0.182,0.198
